In [ ]:
from pathlib import Path
import pandas as pd
import re


def create_temperature_xyz_dataset(op_name):
    """
    Beispiel:
        create_temperature_xyz_dataset("OP03")

    Ausgabe:
        processedData/OP03_temperature_xyz.csv

    Format:
        op,type,t,x,y,z,T
    """

    root = Path(".")

    output_dir = root / "processedData"
    output_dir.mkdir(exist_ok=True)

    coord_files = {
        "cc": root / "coordinates" / "Coordinates - Grid Cell Center.csv",
        "g": root / "coordinates" / "Coordinates - Grid Gehäusewand.csv",
        "jr1c": root / "coordinates" / "Coordinates - Grid JR1 Center.csv"
    }

    def load_coordinates(file_path):

        df = pd.read_csv(file_path)

        rename = {}

        for col in df.columns:
            col_lower = col.lower()

            if "position[x]" in col_lower:
                rename[col] = "x"
            elif "position[y]" in col_lower:
                rename[col] = "y"
            elif "position[z]" in col_lower:
                rename[col] = "z"

        df = df.rename(columns=rename)

        return df[["x", "y", "z"]]

    coordinates = {
        k: load_coordinates(v)
        for k, v in coord_files.items()
    }

    op_folder = root / "storeOPs" / op_name

    if not op_folder.exists():
        raise FileNotFoundError(f"Ordner nicht gefunden: {op_folder}")

    all_rows = []

    csv_files = list(op_folder.glob("*T_grid*.csv"))

    print(f"Gefundene T_grid Dateien: {len(csv_files)}")

    for csv_file in csv_files:

        file_name = csv_file.name.lower()

        if "t_grid_cc" in file_name:
            grid_type = "cc"
            monitor_pattern = r"cc_(\d+)"
        elif "t_grid_g" in file_name:
            grid_type = "g"
            monitor_pattern = r"g_(\d+)"
        elif "t_grid_jr1c" in file_name:
            grid_type = "jr1c"
            monitor_pattern = r"jr1c_(\d+)"
        else:
            continue

        print(f"Bearbeite: {csv_file.name}")

        temp_df = pd.read_csv(csv_file)

        time_col = temp_df.columns[0]

        coord_df = coordinates[grid_type]

        for column in temp_df.columns[1:]:

            match = re.search(monitor_pattern, column)

            if not match:
                continue

            monitor_idx = int(match.group(1)) - 1

            if monitor_idx >= len(coord_df):
                continue

            xyz = coord_df.iloc[monitor_idx]

            tmp = pd.DataFrame({
                "op": op_name,
                "type": grid_type,
                "t": temp_df[time_col],
                "x": xyz["x"],
                "y": xyz["y"],
                "z": xyz["z"],
                "T": temp_df[column]
            })

            all_rows.append(tmp)

    if len(all_rows) == 0:
        raise ValueError(
            f"Keine Monitor-Spalten in {op_name} gefunden."
        )

    result_df = pd.concat(all_rows, ignore_index=True)

    output_file = output_dir / f"{op_name}_temperature_xyz.csv"

    result_df.to_csv(output_file, index=False)

    print()
    print(f"Gespeichert: {output_file}")
    print(f"Zeilen: {len(result_df):,}")

    return result_df

In [ ]:
df = create_temperature_xyz_dataset("OP03")

df.head()